In [1]:
import sys
sys.path.insert(0, '../gofher')

import os
import matplotlib.image as mpimg
import numpy as np

from gofher import run_gofher,run_gofher_with_parameters
from visualize import visualize
from file_helper import write_csv,check_if_folder_exists_and_create
from spin_parity import read_spin_parity_galaxies_label_from_csv, standardize_galaxy_name
from sparcfire import read_sparcfire_galaxy_csv, get_ref_band_and_gofher_params

In [2]:
survery_to_use = "sdss" #Note: for sdss we are not using u do to poor quality

BANDS_IN_ORDER = ['g','r','i','z'] #Important: Must stay in order of BLUEST to REDDEST Waveband (Editting this will cause gofher to no longer correctly evaluate redder side of galaxy)
REF_BANDS_IN_ORDER = ['r','i','z','g'] #The prefernce each waveband being choosen as refernce band from highest priority to lowest priority

In [ ]:
#figure_to_run_on = "figure8"
figure_to_run_on = ["figure11","figure10","figure8","figure9"]
#figure_to_run_on = ["figure8","figure9","figure10","figure11"]
types_of_runs = ["inital","fixed-center","fixed"]

steps = 4
bulge_disk_fs = np.linspace(0.0,1.0,steps+1)

#panstarrs:
# = None #None or a positive integer

#sdss:
#NOTE: sdss needs to flip color image
bin_size = 4 #should be 4
#Source: "The median seeing of all SDSS imaging data (using the psfWidth metric) is 1.32 arcseconds in the r-band."
#"The pixel size in the Sloan Digital Sky Survey (SDSS) is 0.396 arcseconds per pixel" - https://classic.sdss.org/dr3/instruments/imager/
bin_prior_to_param_fitting = True

In [4]:
generate_verbose_csv = True #True
generate_ebm_csv = True
generate_params_csv = True #True
generate_visualization = True
save_visualization = True

In [5]:
#Important: Make sure you update these values:
path_to_catalog_data = "..\\..\\spin-parity-catalog-data"
#path_to_output = "..\\..\\gofher-data\\panstarrs\\default_ellipse_mask_fitting" #- https://www.sdss4.org/dr17/imaging/other_info/
#path_to_output = "E:\\grad_school\\research\\spring_2025\\sdss\\default_ellipse_mask_fitting"

def get_path_to_output(figure_to_run_on,run_type="",bulge_disk_f=1.0):
    #TODO check type of 
    path_to_output_base = os.path.join("E:\\grad_school\\research\\spring_2025",survery_to_use,run_type)
    folder = run_type

    if run_type != "":
        folder += "_{}".format(str(bulge_disk_f).replace(".","_"))
    
    return os.path.join(path_to_output_base,folder)

def get_visulization_save_path_folder(figure_to_run_on,run_type="",bulge_disk_f=1.0):
    path_to_output_base = os.path.join("E:\\grad_school\\research\\spring_2025",survery_to_use,run_type)
    folder = run_type

    if run_type != "":
        folder += "_{}".format(str(bulge_disk_f).replace(".","_"))
    
    return os.path.join(path_to_output_base,folder,figure_to_run_on)

In [6]:
def get_fits_path(name,band,figure_to_run_on):
    """the file path of where existing fits files can be found"""
    return os.path.join(path_to_catalog_data,survery_to_use,figure_to_run_on,name,"{}_{}.fits".format(name,band))

def get_color_image_path(name,figure_to_run_on):
    file_type = "png"
    if survery_to_use == "panstarrs": file_type = "jfif"
    return os.path.join(path_to_catalog_data,survery_to_use,figure_to_run_on,name,"{}_color.{}".format(name,file_type))

def get_path_to_catalog_csv(figure_to_run_on):
    return os.path.join(path_to_catalog_data,"catalog","{}.csv".format(figure_to_run_on))

In [7]:
def get_paper_dark_side_labels(figure_to_run_on):
    return read_spin_parity_galaxies_label_from_csv(get_path_to_catalog_csv(figure_to_run_on))

def get_galaxies(figure_to_run_on):
    return os.listdir(os.path.join(path_to_catalog_data,survery_to_use,figure_to_run_on))

def get_sparcfire_path(figure_to_run_on): #temp
    table_dict = {"figure8":"2","figure9":"3","figure10":"4","figure11":"5"}
    pa_key = table_dict.get(figure_to_run_on,"")
    if pa_key == "": raise ValueError("Invlid figure to run on")

    return "C:\\Users\\school\\Desktop\\cross_id\\sdss_mosaic_construction\\SpArcFiRe_output\\table{}\\G.out\\galaxy.csv".format(pa_key)

In [8]:
def _ensure_path_exists(path_to_output,make_ouput_folder_if_not_exists=True):
    if not make_ouput_folder_if_not_exists:
        raise ValueError("The path output is not found {} - make sure you update path_to_output".format(path_to_output))
    
    if not os.path.exists(path_to_output):
        os.makedirs(path_to_output)

In [ ]:
def run_gofher_on_catalog(figure_to_run_on,run_type="",bulge_disk_f=1.0):
    paper_labels = get_paper_dark_side_labels(figure_to_run_on)

    if run_type != "":
        sparcfire_gals = read_sparcfire_galaxy_csv(get_sparcfire_path(figure_to_run_on))

    verbose_header = []
    verbose_rows = []

    ebm_header = []
    ebm_rows = []

    params_header = []
    params_rows = []

    i = 1

    path_to_output = get_path_to_output(figure_to_run_on,run_type=run_type,bulge_disk_f=bulge_disk_f)
    _ensure_path_exists(path_to_output)

    def get_fits_path_cat(name,band):
        """the file path of where existing fits files can be found"""
        return get_fits_path(name,band,figure_to_run_on)

    for name in get_galaxies(figure_to_run_on):

        if standardize_galaxy_name(name) not in paper_labels:
            print("skipping",name)
            continue

        print(name, i,"of",len(get_galaxies(figure_to_run_on)))

        try:
            paper_label = paper_labels[standardize_galaxy_name(name)]
            
            if run_type == "":
                gal = run_gofher(name,get_fits_path_cat,BANDS_IN_ORDER,REF_BANDS_IN_ORDER, paper_label,s=bin_size,bin_prior_to_param_fitting=bin_prior_to_param_fitting)
            else:
                ref_band, inital_gofher_params = get_ref_band_and_gofher_params(sparcfire_gals[name],REF_BANDS_IN_ORDER,bulge_disk_f)
                gal = run_gofher_with_parameters(name,get_fits_path_cat,BANDS_IN_ORDER,ref_band,inital_gofher_params,paper_label=paper_label,mode=run_type)


            if generate_verbose_csv:
                (header,row) = gal.get_verbose_csv_header_and_row(BANDS_IN_ORDER,paper_label)
                if len(verbose_header) == 0: verbose_header = header
                verbose_rows.append(row)

            if generate_ebm_csv:
                (header,row) = gal.get_ebm_csv_header_and_row(BANDS_IN_ORDER, paper_label)
                if len(ebm_header) == 0: ebm_header = header
                ebm_rows.append(row)

            if generate_params_csv:
                (header,row) = gal.get_params_csv_header_and_row()
                if len(params_header) == 0: params_header = header
                params_rows.append(row)

            if generate_visualization:
                save_path = ''
                
                if save_visualization:
                    sub_folder = get_visulization_save_path_folder(figure_to_run_on,run_type,bulge_disk_f)
                    check_if_folder_exists_and_create(sub_folder)
                    save_path = os.path.join(sub_folder,"{}.png".format(name))

                color_image = color = mpimg.imread(get_color_image_path(name,figure_to_run_on))
                visual_string = "type={} f={}".format(run_type,bulge_disk_f)
                visualize(gal,color_image,BANDS_IN_ORDER,paper_label,save_path=save_path,color_flip=(survery_to_use=="sdss"),show_stats=True,visual_string=visual_string)
        except Exception as e:
            print(e)
        i += 1
        
    if generate_verbose_csv:
        verbose_csv_path = os.path.join(path_to_output,"{}_verbose.csv".format(figure_to_run_on))
        write_csv(verbose_csv_path,verbose_header,verbose_rows)

    if generate_ebm_csv:
        ebm_csv_path = os.path.join(path_to_output,"{}_ebm.csv".format(figure_to_run_on))
        write_csv(ebm_csv_path,ebm_header,ebm_rows)

    if generate_params_csv:
        params_csv_path = os.path.join(path_to_output,"{}_params.csv".format(figure_to_run_on))
        write_csv(params_csv_path,params_header,params_rows)



In [ ]:
if not os.path.exists(path_to_catalog_data):
    raise ValueError("The path to the catalog is not found {} - make sure you update path_to_catalog_data".format(path_to_catalog_data))

for figure_to_run_on in figure_to_run_on:
    print("figure",figure_to_run_on)
    for run_type in types_of_runs:
        print("run type",run_type)
        for bulge_disk_f in bulge_disk_fs:
            run_gofher_on_catalog(figure_to_run_on,run_type=run_type,bulge_disk_f=bulge_disk_f)

IC1683 1 of 114
IC1683 1 of 114
IC1683 1 of 114
IC1683 1 of 114
IC1683 1 of 114
IC1683 1 of 114
IC1683 1 of 114
IC1683 1 of 114
IC1683 1 of 114
IC1683 1 of 114
IC1683 1 of 114
IC1683 1 of 114
IC1683 1 of 114
IC1683 1 of 114
IC1683 1 of 114
